# 12a — Validation Evaluation

This notebook answers the Week 6 validation requirements directly while keeping validation fully consistent with the tuned Week 5 models.

### Tasks
- Load the tuned model definitions and hyperparameters from Week 5 CSV outputs.
- Recreate the tuned classifiers, including XGBoost.
- Restore the Week 5 feature-selection setting (`feature_selection__k`).
- Reuse the shared leakage-safe preprocessing/evaluation framework in `src/modeling`.
- Fit the tuned pipeline on training data only.
- Evaluate the models on the held-out validation data.
- Save predicted classes and predicted probabilities.
- Compare validation performance.
- Generate confusion matrices.
- Select the best model(s).

### Required outputs
- `outputs/metrics/validation_results.csv`
- `outputs/metrics/validation_predictions.csv`
- `outputs/metrics/validation_probabilities.csv`
- validation confusion matrices
- validation comparison table
- validation comparison figure
- best-model summary and justification

> Week 5 saved the tuned settings rather than fitted estimators. This notebook rebuilds the same tuned pipeline configuration and evaluates it on held-out validation data.


## 1. Setup

In [111]:
from pathlib import Path
import ast
import json
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is required because Week 5 includes tuned XGBoost models. "
        "Install it with `pip install xgboost`."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the project repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reuse the shared modeling framework instead of rebuilding preprocessing here.
from src.modeling.preprocessing import prepare_dataset
from src.modeling.evaluation import calculate_metrics

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for folder in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Using shared preprocessing from src/modeling/preprocessing.py")
print("Using shared metrics from src/modeling/evaluation.py")


Project root: C:\Users\bramw\OneDrive - GUSCanada\Desktop\Semester 5\Capstone Project\AI-Assisted-Screening-of-Parkinson-s-Disease
Using shared preprocessing from src/modeling/preprocessing.py
Using shared metrics from src/modeling/evaluation.py


## 2. Load the Week 5 tuning outputs

In [112]:
week5_files = {
    "candidate_models": METRICS_DIR / "candidate_models.csv",
    "tuned_models": METRICS_DIR / "tuned_models.csv",
    "hyperparameter_tuning_summary": METRICS_DIR / "hyperparameter_tuning_summary.csv",
}

week5_tables = {}

for name, path in week5_files.items():
    if path.exists():
        week5_tables[name] = pd.read_csv(path)
        print(f"✓ Loaded {path.name}: {len(week5_tables[name])} row(s)")
    else:
        print(f"— Not found: {path.name}")

if not week5_tables:
    raise FileNotFoundError(
        "No Week 5 tuning CSV files were found in outputs/metrics."
    )

for name, table in week5_tables.items():
    print(f"\n{name}.csv")
    display(table)

✓ Loaded candidate_models.csv: 9 row(s)
✓ Loaded tuned_models.csv: 9 row(s)
✓ Loaded hyperparameter_tuning_summary.csv: 9 row(s)

candidate_models.csv


,Dataset,Model,Macro F1,Balanced Accuracy,Accuracy,Precision Macro,Recall Macro,Training Time (s),Best Parameters,Inference Time (s),Model Size (KB),Standard Deviation,Coefficient of Variation (%)
0,Wearable + Questionnaire,Logistic Regression,0.687465,0.689042,0.743776,0.695821,0.689042,102.895731,"{'classifier__C': 1, 'classifier__max_iter': 1...",0.279736,268.840820,0.070230,10.215746
1,Full Multimodal,Random Forest,0.686305,0.693289,0.737762,0.688910,0.693289,7850.438869,"{'classifier__max_depth': None, 'classifier__m...",0.236057,1780.513672,0.056373,8.213921
2,Full Multimodal,Logistic Regression,0.685731,0.689227,0.740746,0.694050,0.689227,103.903555,"{'classifier__C': 1, 'classifier__max_iter': 1...",0.278383,269.896484,0.065021,9.482007
3,Wearable + Questionnaire,XGBoost,0.684395,0.667196,0.762005,0.755484,0.667196,46294.753700,"{'classifier__colsample_bytree': 0.8, 'classif...",0.900451,836.027344,0.050233,7.339758
4,Wearable + Questionnaire,Random Forest,0.683423,0.684023,0.737669,0.691162,0.684023,7339.940151,"{'classifier__max_depth': None, 'classifier__m...",0.238205,2670.425781,0.051214,7.493693
5,Full Multimodal,XGBoost,0.676227,0.659684,0.765035,0.767604,0.659684,19850.267780,"{'classifier__colsample_bytree': 1.0, 'classif...",0.172930,835.943359,0.045293,6.697960
6,Demographics + Questionnaire,Random Forest,0.619851,0.643992,0.698322,0.636911,0.643992,1507.516596,"{'classifier__max_depth': None, 'classifier__m...",0.101763,706.986328,0.059567,9.609972
7,Demographics + Questionnaire,Logistic Regression,0.608928,0.657845,0.655571,0.606712,0.657845,27.903671,"{'classifier__C': 0.1, 'classifier__max_iter':...",0.038431,8.420898,0.040426,6.638864
8,Demographics + Questionnaire,XGBoost,0.598170,0.592890,0.692214,0.645921,0.592890,222.591484,"{'classifier__colsample_bytree': 1.0, 'classif...",0.060958,647.409180,0.058809,9.831443



tuned_models.csv


,Dataset,Model,Filename
0,Demographics + Questionnaire,Logistic Regression,demographics_plus_questionnaire_logistic_regre...
1,Demographics + Questionnaire,Random Forest,demographics_plus_questionnaire_random_forest....
2,Demographics + Questionnaire,XGBoost,demographics_plus_questionnaire_xgboost.joblib
3,Wearable + Questionnaire,Logistic Regression,wearable_plus_questionnaire_logistic_regressio...
4,Wearable + Questionnaire,Random Forest,wearable_plus_questionnaire_random_forest.joblib
5,Wearable + Questionnaire,XGBoost,wearable_plus_questionnaire_xgboost.joblib
6,Full Multimodal,Logistic Regression,full_multimodal_logistic_regression.joblib
7,Full Multimodal,Random Forest,full_multimodal_random_forest.joblib
8,Full Multimodal,XGBoost,full_multimodal_xgboost.joblib



hyperparameter_tuning_summary.csv


,Dataset,Model,Macro F1,Balanced Accuracy,Accuracy,Precision Macro,Recall Macro,Training Time (s),Best Parameters
0,Demographics + Questionnaire,Logistic Regression,0.608928,0.657845,0.655571,0.606712,0.657845,27.903671,"{'classifier__C': 0.1, 'classifier__max_iter':..."
1,Demographics + Questionnaire,Random Forest,0.619851,0.643992,0.698322,0.636911,0.643992,1507.516596,"{'classifier__max_depth': None, 'classifier__m..."
2,Demographics + Questionnaire,XGBoost,0.598170,0.592890,0.692214,0.645921,0.592890,222.591484,"{'classifier__colsample_bytree': 1.0, 'classif..."
3,Wearable + Questionnaire,Logistic Regression,0.687465,0.689042,0.743776,0.695821,0.689042,102.895731,"{'classifier__C': 1, 'classifier__max_iter': 1..."
4,Wearable + Questionnaire,Random Forest,0.683423,0.684023,0.737669,0.691162,0.684023,7339.940151,"{'classifier__max_depth': None, 'classifier__m..."
5,Wearable + Questionnaire,XGBoost,0.684395,0.667196,0.762005,0.755484,0.667196,46294.753700,"{'classifier__colsample_bytree': 0.8, 'classif..."
6,Full Multimodal,Logistic Regression,0.685731,0.689227,0.740746,0.694050,0.689227,103.903555,"{'classifier__C': 1, 'classifier__max_iter': 1..."
7,Full Multimodal,Random Forest,0.686305,0.693289,0.737762,0.688910,0.693289,7850.438869,"{'classifier__max_depth': None, 'classifier__m..."
8,Full Multimodal,XGBoost,0.676227,0.659684,0.765035,0.767604,0.659684,19850.267780,"{'classifier__colsample_bytree': 1.0, 'classif..."


## 3. Build the tuned-model table

Hyperparameters are loaded from the Week 5 output that actually contains the saved **Best Parameters** column.

Priority:
1. `candidate_models.csv`
2. `hyperparameter_tuning_summary.csv`

`tuned_models.csv` is used only as optional model-file metadata because it may contain model names and filenames without the tuned parameter dictionaries.


In [113]:
def normalize_col_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower()
    ).strip("_")


def find_col(df, choices):
    normalized = {
        normalize_col_name(c): c
        for c in df.columns
    }

    for choice in choices:
        key = normalize_col_name(choice)
        if key in normalized:
            return normalized[key]

    return None


def parse_params(value):
    if value is None or (
        isinstance(value, float)
        and pd.isna(value)
    ):
        return {}

    if isinstance(value, dict):
        return value

    text = str(value).strip()

    if not text:
        return {}

    for parser in (ast.literal_eval, json.loads):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    raise ValueError(
        f"Could not parse saved parameter dictionary: {text[:120]}"
    )


def standardize_model_table(df, require_params=False):
    model_col = find_col(
        df,
        [
            "model",
            "model_name",
            "classifier",
            "estimator",
            "candidate_model",
        ],
    )

    dataset_col = find_col(
        df,
        [
            "dataset",
            "dataset_name",
            "modality",
            "feature_set",
        ],
    )

    params_col = find_col(
        df,
        [
            "Best Parameters",
            "best_params",
            "best_parameters",
            "parameters",
            "params",
            "hyperparameters",
            "tuned_parameters",
        ],
    )

    filename_col = find_col(
        df,
        [
            "filename",
            "file",
            "model_file",
            "model_filename",
        ],
    )

    if model_col is None:
        raise ValueError(
            f"No model-name column found. Columns: {list(df.columns)}"
        )

    if require_params and params_col is None:
        raise ValueError(
            f"No Best Parameters column found. Columns: {list(df.columns)}"
        )

    result = pd.DataFrame({
        "dataset": (
            df[dataset_col].astype(str)
            if dataset_col
            else "multimodal_full"
        ),
        "model": df[model_col].astype(str),
    })

    if params_col:
        result["parameters"] = df[params_col].apply(parse_params)

    if filename_col:
        result["filename"] = df[filename_col].astype(str)

    return result


parameter_source_name = None
parameter_table = None

for preferred in [
    "candidate_models",
    "hyperparameter_tuning_summary",
]:
    if (
        preferred not in week5_tables
        or week5_tables[preferred].empty
    ):
        continue

    try:
        candidate = standardize_model_table(
            week5_tables[preferred],
            require_params=True,
        )

        if candidate["parameters"].map(bool).all():
            parameter_source_name = preferred
            parameter_table = candidate
            break

    except ValueError:
        continue


if parameter_table is None:
    raise ValueError(
        "Could not recover non-empty Week 5 hyperparameters from "
        "candidate_models.csv or hyperparameter_tuning_summary.csv. "
        "Validation will not continue with empty {} parameter dictionaries."
    )


tuned_models = parameter_table.copy()

# Join optional filenames separately.
if (
    "tuned_models" in week5_tables
    and not week5_tables["tuned_models"].empty
):
    inventory = standardize_model_table(
        week5_tables["tuned_models"]
    )

    if "filename" in inventory.columns:
        tuned_models = tuned_models.merge(
            inventory[
                ["dataset", "model", "filename"]
            ].drop_duplicates(),
            on=["dataset", "model"],
            how="left",
        )


tuned_models = tuned_models.drop_duplicates(
    subset=["dataset", "model"]
).reset_index(drop=True)

print(
    f"Using hyperparameters from: "
    f"{parameter_source_name}.csv"
)
print(
    "Empty parameter dictionaries:",
    int(
        (~tuned_models["parameters"].map(bool)).sum()
    ),
)

display(tuned_models)


Using hyperparameters from: candidate_models.csv
Empty parameter dictionaries: 0


,dataset,model,parameters,filename
0,Wearable + Questionnaire,Logistic Regression,"{'classifier__C': 1, 'classifier__max_iter': 1...",wearable_plus_questionnaire_logistic_regressio...
1,Full Multimodal,Random Forest,"{'classifier__max_depth': None, 'classifier__m...",full_multimodal_random_forest.joblib
2,Full Multimodal,Logistic Regression,"{'classifier__C': 1, 'classifier__max_iter': 1...",full_multimodal_logistic_regression.joblib
3,Wearable + Questionnaire,XGBoost,"{'classifier__colsample_bytree': 0.8, 'classif...",wearable_plus_questionnaire_xgboost.joblib
4,Wearable + Questionnaire,Random Forest,"{'classifier__max_depth': None, 'classifier__m...",wearable_plus_questionnaire_random_forest.joblib
5,Full Multimodal,XGBoost,"{'classifier__colsample_bytree': 1.0, 'classif...",full_multimodal_xgboost.joblib
6,Demographics + Questionnaire,Random Forest,"{'classifier__max_depth': None, 'classifier__m...",demographics_plus_questionnaire_random_forest....
7,Demographics + Questionnaire,Logistic Regression,"{'classifier__C': 0.1, 'classifier__max_iter':...",demographics_plus_questionnaire_logistic_regre...
8,Demographics + Questionnaire,XGBoost,"{'classifier__colsample_bytree': 1.0, 'classif...",demographics_plus_questionnaire_xgboost.joblib


## 4. Find the training and validation datasets

In [114]:
# Optional manual overrides.
# Only fill these in if automatic discovery does not find the correct files.
MANUAL_DATA_FILES = {
    # Example:
    # "multimodal_full": {
    #     "train": DATA_DIR / "processed" / "multimodal_full_train.csv",
    #     "validation": DATA_DIR / "processed" / "multimodal_full_validation.csv",
    # }
}

all_csvs = []

for root in [DATA_DIR, OUTPUTS_DIR]:
    if root.exists():
        all_csvs.extend(root.rglob("*.csv"))

def clean_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def choose_file(dataset, kind):
    # Manual override wins.
    if dataset in MANUAL_DATA_FILES:
        value = MANUAL_DATA_FILES[dataset].get(kind)
        if value:
            p = Path(value)
            return p if p.exists() else None

    ds = clean_text(dataset)

    kind_words = {
        "train": ["train", "training"],
        "validation": ["validation", "valid", "val"],
    }[kind]

    candidates = []

    for p in all_csvs:
        name = clean_text(p.stem)

        # Ignore Week 5/Week 6 metric outputs.
        if any(
            bad in name
            for bad in [
                "candidate_models",
                "tuned_models",
                "hyperparameter_tuning",
                "validation_results",
                "confusion_matrix",
                "performance_summary",
                "metrics",
            ]
        ):
            continue

        if not any(word in name for word in kind_words):
            continue

        score = 0

        if ds and ds in name:
            score += 100

        ds_tokens = set(ds.split("_"))
        name_tokens = set(name.split("_"))
        score += len(ds_tokens & name_tokens)

        candidates.append((score, p))

    candidates.sort(key=lambda x: x[0], reverse=True)

    if candidates:
        return candidates[0][1]

    return None

data_records = []

for dataset in tuned_models["dataset"].unique():
    data_records.append({
        "dataset": dataset,
        "train_file": choose_file(dataset, "train"),
        "validation_file": choose_file(dataset, "validation"),
    })

data_inventory = pd.DataFrame(data_records)

display(data_inventory)

missing = data_inventory[
    data_inventory["train_file"].isna()
    | data_inventory["validation_file"].isna()
]

if not missing.empty:
    print("\nAutomatic discovery could not find every required train/validation file.")
    print("Available CSV files that contain 'train' or 'valid':")

    for p in all_csvs:
        n = p.name.lower()
        if "train" in n or "valid" in n:
            print(" -", p.relative_to(PROJECT_ROOT))

    print(
        "\nIf the correct files are listed above, add their paths to "
        "MANUAL_DATA_FILES and rerun this section."
    )

,dataset,train_file,validation_file
0,Wearable + Questionnaire,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...
1,Full Multimodal,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...
2,Demographics + Questionnaire,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...


## 5. Recreate the tuned models

The classifier parameters and feature-selection parameter are recovered separately.

`feature_selection__k` remains a pipeline setting and is restored before the classifier. The preprocessing stage is generated by `src.modeling.preprocessing.prepare_dataset`, ensuring validation uses the same shared leakage-safe preprocessing rather than a notebook-specific copy.


In [115]:
FEATURE_SELECTION_KEYS = [
    "feature_selection__k",
    "select__k",
    "selector__k",
    "k",
    "n_selected_features",
]


def strip_classifier_prefix(key):
    key = str(key)

    for prefix in [
        "classifier__",
        "model__",
        "estimator__",
        "clf__",
    ]:
        if key.startswith(prefix):
            return key[len(prefix):]

    return key


def get_selected_k(params):
    for key in FEATURE_SELECTION_KEYS:
        if key in params:
            value = params[key]

            if (
                isinstance(value, str)
                and value.lower() == "all"
            ):
                return "all"

            return int(value)

    return None


def classifier_params_only(params):
    cleaned = {}

    for key, value in params.items():
        if key in FEATURE_SELECTION_KEYS:
            continue

        if str(key).startswith(
            ("preprocessing__", "preprocess__")
        ):
            continue

        cleaned[
            strip_classifier_prefix(key)
        ] = value

    return cleaned


def make_classifier(model_name, params):
    name = clean_text(model_name)
    params = classifier_params_only(params)

    if "dummy" in name:
        return DummyClassifier(**params)

    if "logistic" in name:
        params.setdefault("max_iter", 5000)
        return LogisticRegression(**params)

    if (
        "random_forest" in name
        or "randomforest" in name
    ):
        params.setdefault("random_state", 42)
        return RandomForestClassifier(**params)

    if (
        "xgboost" in name
        or name in {"xgb", "xgbclassifier"}
    ):
        params.setdefault("random_state", 42)
        params.setdefault("n_jobs", -1)
        params.setdefault(
            "objective",
            "multi:softprob",
        )
        params.setdefault(
            "eval_metric",
            "mlogloss",
        )
        return XGBClassifier(**params)

    if (
        "extra_trees" in name
        or "extratrees" in name
    ):
        params.setdefault("random_state", 42)
        return ExtraTreesClassifier(**params)

    if (
        "decision_tree" in name
        or name in {"tree", "dt"}
    ):
        params.setdefault("random_state", 42)
        return DecisionTreeClassifier(**params)

    if (
        "hist_gradient" in name
        or "histgradient" in name
    ):
        params.setdefault("random_state", 42)
        return HistGradientBoostingClassifier(**params)

    if (
        "gradient_boost" in name
        or "gradientboost" in name
    ):
        params.setdefault("random_state", 42)
        return GradientBoostingClassifier(**params)

    if (
        name in {
            "svc",
            "svm",
            "support_vector_machine",
        }
        or "support_vector" in name
    ):
        params.setdefault("probability", True)
        return SVC(**params)

    if (
        "knn" in name
        or "nearest_neighbor" in name
    ):
        return KNeighborsClassifier(**params)

    raise ValueError(
        f"Model '{model_name}' is not mapped "
        "in make_classifier()."
    )


feature_selection_audit = tuned_models[
    ["dataset", "model", "parameters"]
].copy()

feature_selection_audit[
    "feature_selection__k"
] = feature_selection_audit[
    "parameters"
].apply(get_selected_k)

display(
    feature_selection_audit[
        [
            "dataset",
            "model",
            "feature_selection__k",
        ]
    ]
)


,dataset,model,feature_selection__k
0,Wearable + Questionnaire,Logistic Regression,1000
1,Full Multimodal,Random Forest,50
2,Full Multimodal,Logistic Regression,1000
3,Wearable + Questionnaire,XGBoost,1000
4,Wearable + Questionnaire,Random Forest,50
5,Full Multimodal,XGBoost,1000
6,Demographics + Questionnaire,Random Forest,50
7,Demographics + Questionnaire,Logistic Regression,all
8,Demographics + Questionnaire,XGBoost,all


## 6. Fit on training data and evaluate on validation

For every tuned Week 5 candidate:

1. the shared `prepare_dataset()` function identifies the modeling features and creates the shared preprocessing pipeline;
2. the saved Week 5 `feature_selection__k` is restored;
3. the tuned classifier is recreated with the saved classifier hyperparameters;
4. preprocessing, feature selection, and the classifier are fitted on **training data only**;
5. validation data are used only for predictions and evaluation;
6. both predicted classes and `predict_proba()` outputs are saved for later Week 6 analyses.


In [116]:
results = []
prediction_store = {}
prediction_rows = []
probability_rows = []


for _, model_row in tuned_models.iterrows():
    dataset = model_row["dataset"]
    model_name = model_row["model"]
    params = model_row["parameters"]

    data_row = data_inventory[
        data_inventory["dataset"] == dataset
    ]

    if data_row.empty:
        print(
            f"✗ {dataset} | {model_name}: "
            "no dataset mapping"
        )
        continue

    train_file = data_row.iloc[0]["train_file"]
    validation_file = (
        data_row.iloc[0]["validation_file"]
    )

    if (
        pd.isna(train_file)
        or pd.isna(validation_file)
    ):
        print(
            f"✗ {dataset} | {model_name}: "
            "train/validation file missing"
        )
        continue

    try:
        train_df = pd.read_csv(
            train_file,
            dtype={ID_COLUMN: str},
        )

        validation_df = pd.read_csv(
            validation_file,
            dtype={ID_COLUMN: str},
        )

        if TARGET not in train_df.columns:
            raise ValueError(
                f"{TARGET!r} missing "
                "from training data"
            )

        if TARGET not in validation_df.columns:
            raise ValueError(
                f"{TARGET!r} missing "
                "from validation data"
            )

        # Shared project feature selection/exclusion
        # and preprocessing configuration.
        X_train, y_train, preprocessing = (
            prepare_dataset(train_df)
        )

        X_val, y_val, _ = (
            prepare_dataset(validation_df)
        )

        # Align validation columns to the exact
        # training feature schema.
        missing_in_validation = [
            c
            for c in X_train.columns
            if c not in X_val.columns
        ]

        if missing_in_validation:
            raise ValueError(
                "Validation data are missing "
                f"{len(missing_in_validation)} "
                "training feature(s): "
                f"{missing_in_validation[:10]}"
            )

        X_val = X_val[
            X_train.columns
        ].copy()

        classifier = make_classifier(
            model_name,
            params,
        )

        selected_k = get_selected_k(params)

        pipeline_steps = [
            (
                "preprocessing",
                preprocessing,
            )
        ]

        if selected_k is not None:
            pipeline_steps.append(
                (
                    "feature_selection",
                    SelectKBest(
                        score_func=f_classif,
                        k=selected_k,
                    ),
                )
            )

        pipeline_steps.append(
            (
                "classifier",
                classifier,
            )
        )

        pipeline = Pipeline(
            pipeline_steps
        )

        # Training information only.
        pipeline.fit(
            X_train,
            y_train,
        )

        y_pred = pipeline.predict(X_val)

        metrics = calculate_metrics(
            y_val,
            y_pred,
        )

        key = (
            f"{clean_text(dataset)}"
            f"__{clean_text(model_name)}"
        )

        prediction_store[key] = (
            y_val.copy(),
            y_pred.copy(),
        )

        results.append({
            "result_key": key,
            "dataset": dataset,
            "model": model_name,
            "n_validation": len(y_val),
            "feature_selection__k": selected_k,
            **metrics,
        })

        if ID_COLUMN in validation_df.columns:
            participant_ids = (
                validation_df[
                    ID_COLUMN
                ].astype(str).values
            )
        else:
            participant_ids = (
                np.arange(
                    len(validation_df)
                ).astype(str)
            )

        for (
            participant_id,
            true_label,
            predicted_label,
        ) in zip(
            participant_ids,
            y_val,
            y_pred,
        ):
            prediction_rows.append({
                ID_COLUMN: participant_id,
                "dataset": dataset,
                "model": model_name,
                "true_label": true_label,
                "predicted_label": predicted_label,
            })

        # Save probabilities for calibration,
        # Brier score, ROC/PR, SHAP, etc.
        if hasattr(
            pipeline,
            "predict_proba",
        ):
            probabilities = (
                pipeline.predict_proba(
                    X_val
                )
            )

            classes = pipeline.classes_

            for row_index, (
                participant_id,
                true_label,
            ) in enumerate(
                zip(
                    participant_ids,
                    y_val,
                )
            ):
                probability_row = {
                    ID_COLUMN: participant_id,
                    "dataset": dataset,
                    "model": model_name,
                    "true_label": true_label,
                }

                for (
                    class_index,
                    class_label,
                ) in enumerate(classes):
                    probability_row[
                        f"prob_class_{class_label}"
                    ] = probabilities[
                        row_index,
                        class_index,
                    ]

                probability_rows.append(
                    probability_row
                )

        else:
            print(
                f"⚠ {dataset} | {model_name}: "
                "predict_proba() unavailable"
            )

        print(
            f"✓ {dataset} | {model_name} | "
            f"k={selected_k} | "
            f"Macro F1="
            f"{metrics['macro_f1']:.4f}"
        )

    except Exception as exc:
        print(
            f"✗ {dataset} | {model_name}: "
            f"{type(exc).__name__}: {exc}"
        )


validation_results = pd.DataFrame(
    results
)

if validation_results.empty:
    raise RuntimeError(
        "No validation model completed "
        "successfully."
    )


validation_results = (
    validation_results
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

validation_results.insert(
    0,
    "rank",
    np.arange(
        1,
        len(validation_results) + 1,
    ),
)


validation_predictions = pd.DataFrame(
    prediction_rows
)

validation_probabilities = pd.DataFrame(
    probability_rows
)


validation_predictions.to_csv(
    METRICS_DIR
    / "validation_predictions.csv",
    index=False,
)

validation_probabilities.to_csv(
    METRICS_DIR
    / "validation_probabilities.csv",
    index=False,
)


print(
    "Saved class-prediction rows:",
    len(validation_predictions),
)

print(
    "Saved probability rows:",
    len(validation_probabilities),
)

display(validation_results)


✓ Wearable + Questionnaire | Logistic Regression | k=1000 | Macro F1=0.6947
✓ Full Multimodal | Random Forest | k=50 | Macro F1=0.7152
✓ Full Multimodal | Logistic Regression | k=1000 | Macro F1=0.6947
✓ Wearable + Questionnaire | XGBoost | k=1000 | Macro F1=0.7098
✓ Wearable + Questionnaire | Random Forest | k=50 | Macro F1=0.6937
✓ Full Multimodal | XGBoost | k=1000 | Macro F1=0.7657
✓ Demographics + Questionnaire | Random Forest | k=50 | Macro F1=0.5193
✓ Demographics + Questionnaire | Logistic Regression | k=all | Macro F1=0.6369
✓ Demographics + Questionnaire | XGBoost | k=all | Macro F1=0.5554
Saved class-prediction rows: 630
Saved probability rows: 630


,rank,result_key,dataset,model,n_validation,feature_selection__k,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,1,full_multimodal__xgboost,Full Multimodal,XGBoost,70,1000,0.785714,0.746931,0.765717,0.794669,0.746931
1,2,full_multimodal__random_forest,Full Multimodal,Random Forest,70,50,0.771429,0.696198,0.715191,0.764564,0.696198
2,3,wearable_questionnaire__xgboost,Wearable + Questionnaire,XGBoost,70,1000,0.757143,0.699546,0.709770,0.733092,0.699546
3,4,wearable_questionnaire__logistic_regression,Wearable + Questionnaire,Logistic Regression,70,1000,0.742857,0.686553,0.694737,0.706878,0.686553
4,5,full_multimodal__logistic_regression,Full Multimodal,Logistic Regression,70,1000,0.742857,0.686553,0.694737,0.706878,0.686553
5,6,wearable_questionnaire__random_forest,Wearable + Questionnaire,Random Forest,70,50,0.757143,0.668420,0.693652,0.753333,0.668420
6,7,demographics_questionnaire__logistic_regression,Demographics + Questionnaire,Logistic Regression,70,all,0.714286,0.624382,0.636947,0.677778,0.624382
7,8,demographics_questionnaire__xgboost,Demographics + Questionnaire,XGBoost,70,all,0.657143,0.557429,0.555382,0.570833,0.557429
8,9,demographics_questionnaire__random_forest,Demographics + Questionnaire,Random Forest,70,50,0.671429,0.514786,0.519274,0.663255,0.514786


## 7. Save `validation_results.csv` and comparison table

In [117]:
validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

comparison_cols = [
    "rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison = validation_results[
    comparison_cols
].copy()

validation_comparison.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

display(validation_comparison.round(4))

,rank,dataset,model,n_validation,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,1,Full Multimodal,XGBoost,70,0.7857,0.7469,0.7657,0.7947,0.7469
1,2,Full Multimodal,Random Forest,70,0.7714,0.6962,0.7152,0.7646,0.6962
2,3,Wearable + Questionnaire,XGBoost,70,0.7571,0.6995,0.7098,0.7331,0.6995
3,4,Wearable + Questionnaire,Logistic Regression,70,0.7429,0.6866,0.6947,0.7069,0.6866
4,5,Full Multimodal,Logistic Regression,70,0.7429,0.6866,0.6947,0.7069,0.6866
5,6,Wearable + Questionnaire,Random Forest,70,0.7571,0.6684,0.6937,0.7533,0.6684
6,7,Demographics + Questionnaire,Logistic Regression,70,0.7143,0.6244,0.6369,0.6778,0.6244
7,8,Demographics + Questionnaire,XGBoost,70,0.6571,0.5574,0.5554,0.5708,0.5574
8,9,Demographics + Questionnaire,Random Forest,70,0.6714,0.5148,0.5193,0.6633,0.5148


## 8. Validation confusion matrices

In [118]:
# Generate and save validation confusion matrices as tables

for _, row in validation_results.iterrows():
    key = row["result_key"]
    y_true, y_pred = prediction_store[key]

    labels = sorted(
        pd.Series(y_true).dropna().unique().tolist()
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {x}" for x in labels],
        columns=[f"Predicted {x}" for x in labels]
    )

    # Save confusion matrix
    csv_path = (
        METRICS_DIR /
        f"{key}_validation_confusion_matrix.csv"
    )

    cm_df.to_csv(csv_path)

    # Show it directly in the notebook
    print(f"\n{row['dataset']} — {row['model']}")
    display(cm_df)

    print(f"Saved: {csv_path.name}")


Full Multimodal — XGBoost


,Predicted 0,Predicted 1,Predicted 2
Actual 0,10,2,0
Actual 1,1,36,4
Actual 2,0,8,9


Saved: full_multimodal__xgboost_validation_confusion_matrix.csv

Full Multimodal — Random Forest


,Predicted 0,Predicted 1,Predicted 2
Actual 0,9,3,0
Actual 1,0,38,3
Actual 2,2,8,7


Saved: full_multimodal__random_forest_validation_confusion_matrix.csv

Wearable + Questionnaire — XGBoost


...

Saved: demographics_questionnaire__logistic_regression_validation_confusion_matrix.csv

Demographics + Questionnaire — XGBoost


,Predicted 0,Predicted 1,Predicted 2
Actual 0,7,3,2
Actual 1,2,35,4
Actual 2,3,10,4


Saved: demographics_questionnaire__xgboost_validation_confusion_matrix.csv

Demographics + Questionnaire — Random Forest


,Predicted 0,Predicted 1,Predicted 2
Actual 0,5,7,0
Actual 1,1,39,1
Actual 2,3,11,3


Saved: demographics_questionnaire__random_forest_validation_confusion_matrix.csv


## 9. Validation comparison figure

In [119]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

plot_df = validation_results[
    ["dataset", "model", "macro_f1", "balanced_accuracy", "accuracy"]
].copy()

plot_df["candidate"] = (
    plot_df["dataset"].astype(str)
    + " | "
    + plot_df["model"].astype(str)
)

plot_df = plot_df.sort_values(
    ["macro_f1", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

figure_path = Path(FIGURES_DIR) / "validation_model_comparison.png"

img = Image.new("RGB", (1200, 850), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default()

draw.text((40, 30), "Validation Model Comparison", fill="black", font=font)
draw.text(
    (40, 55),
    "Models ranked by validation Macro F1. Higher scores are better.",
    fill="black",
    font=font
)

bar_x = 430
bar_width = 650

for i, row in plot_df.iterrows():
    y = 110 + i * 110

    label = f"#{i+1} {row['candidate']}"
    if i == 0:
        label += " - BEST"

    draw.text((40, y), label, fill="black", font=font)

    draw.rectangle(
        [bar_x, y + 25, bar_x + bar_width, y + 50],
        fill="lightgray"
    )

    score_width = int(float(row["macro_f1"]) * bar_width)

    draw.rectangle(
        [bar_x, y + 25, bar_x + score_width, y + 50],
        fill="steelblue"
    )

    metrics = (
        f"Macro F1: {row['macro_f1']:.4f} | "
        f"Balanced Acc: {row['balanced_accuracy']:.4f} | "
        f"Accuracy: {row['accuracy']:.4f}"
    )

    draw.text((40, y + 60), metrics, fill="black", font=font)

img.save(str(figure_path), format="PNG")

print("Saved:", figure_path)
print("Exists:", figure_path.exists())

Saved: C:\Users\bramw\OneDrive - GUSCanada\Desktop\Semester 5\Capstone Project\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\figures\validation_model_comparison.png
Exists: True


## 10. Select the best candidate model(s)

In [120]:
best_by_dataset = (
    validation_results
    .sort_values(
        [
            "dataset",
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[True, False, False],
    )
    .groupby(
        "dataset",
        as_index=False,
    )
    .first()
)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

display(
    best_by_dataset[
        [
            "dataset",
            "model",
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
        ]
    ].round(4)
)

best = validation_results.iloc[0]

print(
    f"Best overall candidate: "
    f"{best['model']} ({best['dataset']})"
)
print(
    f"Macro F1: {best['macro_f1']:.4f}"
)
print(
    f"Balanced accuracy: "
    f"{best['balanced_accuracy']:.4f}"
)

,dataset,model,macro_f1,balanced_accuracy,accuracy
0,Demographics + Questionnaire,Logistic Regression,0.6369,0.6244,0.7143
1,Full Multimodal,XGBoost,0.7657,0.7469,0.7857
2,Wearable + Questionnaire,XGBoost,0.7098,0.6995,0.7571


Best overall candidate: XGBoost (Full Multimodal)
Macro F1: 0.7657
Balanced accuracy: 0.7469


## 11. Validation metrics summary and justification

In [121]:
display(
    validation_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

print("\nJUSTIFICATION")
print(
    f"{best['model']} on {best['dataset']} "
    f"is the strongest candidate to move forward "
    f"because it achieved the highest validation "
    f"Macro F1 ({best['macro_f1']:.4f}). "
    f"Its balanced accuracy was "
    f"{best['balanced_accuracy']:.4f}. "
    f"Macro F1 is used as the primary selection "
    f"metric because it gives equal importance to "
    f"performance across classes, while balanced "
    f"accuracy is used as the secondary comparison."
)

,dataset,model,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Full Multimodal,XGBoost,0.7857,0.7469,0.7657,0.7947,0.7469
1,Full Multimodal,Random Forest,0.7714,0.6962,0.7152,0.7646,0.6962
2,Wearable + Questionnaire,XGBoost,0.7571,0.6995,0.7098,0.7331,0.6995
3,Wearable + Questionnaire,Logistic Regression,0.7429,0.6866,0.6947,0.7069,0.6866
4,Full Multimodal,Logistic Regression,0.7429,0.6866,0.6947,0.7069,0.6866
5,Wearable + Questionnaire,Random Forest,0.7571,0.6684,0.6937,0.7533,0.6684
6,Demographics + Questionnaire,Logistic Regression,0.7143,0.6244,0.6369,0.6778,0.6244
7,Demographics + Questionnaire,XGBoost,0.6571,0.5574,0.5554,0.5708,0.5574
8,Demographics + Questionnaire,Random Forest,0.6714,0.5148,0.5193,0.6633,0.5148



JUSTIFICATION
XGBoost on Full Multimodal is the strongest candidate to move forward because it achieved the highest validation Macro F1 (0.7657). Its balanced accuracy was 0.7469. Macro F1 is used as the primary selection metric because it gives equal importance to performance across classes, while balanced accuracy is used as the secondary comparison.


## 12. Deliverables Check

In [122]:
from pathlib import Path
import pandas as pd

# Required validation deliverables
deliverables = pd.DataFrame([
    {
        "deliverable": "Validation results",
        "path": METRICS_DIR / "validation_results.csv",
    },
    {
        "deliverable": "Validation class predictions",
        "path": METRICS_DIR / "validation_predictions.csv",
    },
    {
        "deliverable": "Validation probability predictions",
        "path": METRICS_DIR / "validation_probabilities.csv",
    },
    {
        "deliverable": "Validation comparison table",
        "path": TABLES_DIR / "validation_comparison.csv",
    },
    {
        "deliverable": "Best candidate table",
        "path": TABLES_DIR / "validation_best_models.csv",
    },
    {
        "deliverable": "Validation comparison figure",
        "path": FIGURES_DIR / "validation_model_comparison.png",
    },
])

deliverables["status"] = deliverables["path"].apply(
    lambda p: "READY" if Path(p).exists() else "MISSING"
)

display(deliverables)

confusion_matrices = list(
    METRICS_DIR.glob("*_validation_confusion_matrix.csv")
)

print(
    f"Validation confusion matrices saved: "
    f"{len(confusion_matrices)}"
)

print(
    "Probability rows saved:",
    len(validation_probabilities),
)

if (deliverables["status"] == "READY").all():
    print()
    print("✓ All required validation outputs are ready.")
else:
    print()
    print("⚠ Some required validation outputs are missing.")

,deliverable,path,status
0,Validation results,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY
1,Validation class predictions,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY
2,Validation probability predictions,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY
3,Validation comparison table,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY
4,Best candidate table,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY
5,Validation comparison figure,C:\Users\bramw\OneDrive - GUSCanada\Desktop\Se...,READY


Validation confusion matrices saved: 18
Probability rows saved: 630

✓ All required validation outputs are ready.
